# Customer Churn Prediction & Analysis

This notebook demonstrates the complete machine learning pipeline for predicting customer churn. It covers data cleaning, exploratory data analysis, feature engineering, model training, and evaluation.

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import sys
import os

warnings.filterwarnings('ignore')
sys.path.insert(0, os.path.abspath('..'))  # Allow importing from src

from sklearn.model_selection import train_test_split
from src.data_cleaning import clean_data, load_data, data_quality_report
from src.feature_engineering import create_features
from src.train_model import build_preprocessor, train_models, NUMERICAL_FEATURES, CATEGORICAL_FEATURES
from src.evaluate_model import evaluate_model, build_comparison_table, plot_confusion_matrices
from src.explain_model import get_feature_names, get_feature_importance, plot_feature_importance

## 2. Load and Inspect Dataset

In [ ]:
raw_path = '../data/customer_churn_messy_raw.csv'
df_raw = load_data(raw_path)
df_raw.head()

## 3. Data-Quality Analysis

In [ ]:
report = data_quality_report(df_raw)
print(f"Missing values:\n{df_raw.isnull().sum()[df_raw.isnull().sum() > 0]}\n")
print(f"Duplicate rows: {report['duplicate_rows']}")

## 4. Cleaning

In [ ]:
# The clean_data function handles duplicates, numerical outliers, and categorical standardization.
df_clean = clean_data(raw_path)
df_clean.head()

## 5. EDA (Exploratory Data Analysis)

In [ ]:
from src.eda import plot_churn_distribution, plot_correlation_heatmap

# Note: The EDA module saves plots to disk. In a standard notebook, you'd just run plt.show().
# We will display the saved images here.
from IPython.display import Image, display

plot_churn_distribution(df_clean)
plot_correlation_heatmap(df_clean)

display(Image(filename='../eda_plots/01_churn_distribution.png'))
display(Image(filename='../eda_plots/11_correlation_heatmap.png'))

## 6. Feature Engineering

In [ ]:
df_features = create_features(df_clean)
df_features[['TenureGroup', 'HighValueCustomer', 'PaymentRisk', 'SupportLevel']].head()

## 7. Train/Test Split

In [ ]:
feature_cols = [c for c in NUMERICAL_FEATURES + CATEGORICAL_FEATURES if c in df_features.columns]
X = df_features[feature_cols]
y = df_features['Churn']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)
print(f"Train size: {len(X_train)} | Test size: {len(X_test)}")

## 8. Model Training & Comparison

In [ ]:
preprocessor = build_preprocessor()
lr_model, rf_model = train_models(X_train, y_train, preprocessor)

## 9. Evaluation

In [ ]:
lr_results = evaluate_model(lr_model, X_test, y_test, "Logistic Regression")
rf_results = evaluate_model(rf_model, X_test, y_test, "Random Forest")

comparison_df = build_comparison_table([lr_results, rf_results])
display(comparison_df)

## 10. Feature Importance

In [ ]:
# Analyze Logistic Regression feature importance (absolute coefficients)
feature_names = get_feature_names(lr_model)
importance_df, method = get_feature_importance(lr_model, feature_names)
plot_feature_importance(importance_df, method, output_dir='../eda_plots')

display(Image(filename='../eda_plots/14_feature_importance.png'))